# 02b m9_pbm Candidate Windows And Physical Features

**Research question.** Can every valid Alpha and Beta candidate window be
represented once, reproducibly, before model variants select different windows?

This notebook is the only expensive physical-feature stage. It generates every
label-free 30-minute to 8-hour candidate, computes F1-F9, writes resumable
substation partitions, and consolidates them into the cache used by Notebooks
02c-02g. It deliberately does **not** reduce each day to the candidate preferred
by an older model.

**Inputs:** final Alpha and Beta data plus the experiment configuration.  
**Outputs:** a local candidate cache, a day-input cache, three compact audit
tables, and a reproducibility manifest.  
**Expected runtime:** approximately 25-40 minutes for a clean run; under one
minute when all validated substation partitions already exist.

## 1. Imports, Paths, And Configuration

The displayed values define the candidate universe. Labels and reviewer
confidence are not passed to candidate generation or F1-F9 calculation. They
remain only in the one-row-per-day audit used to prove a complete later join.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    consolidate_parquet_files,
    load_dataset,
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    to_day_arrays,
    validate_input_hashes,
    write_csv,
    write_manifest,
    write_parquet,
)
from _m9_pbm_features import (  # noqa: E402
    FEATURE_COLUMNS,
    CandidateSpec,
    build_substation_candidate_features,
    compute_candidate_features,
    substation_solar_scale,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SPEC = CandidateSpec.from_config(CONFIG)
SLUG = "02b_m9_pbm_candidate_features"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)
PARTITION_DIR = OUTPUT_DIRS["intermediate"] / "_partitions"
PARTITION_DIR.mkdir(parents=True, exist_ok=True)

display(pd.Series(CONFIG["m9_pbm"]["candidate_windows"], name="candidate_windows"))
display(pd.Series(CONFIG["m9_pbm"]["features"], name="features"))

slots_per_day              96
slot_minutes               15
scan_start_slot            24
scan_end_slot              72
min_duration_slots          2
max_duration_slots         32
shoulder_slots              3
anchor_offset_slots         1
solar_peak_radius_slots    14
max_internal_gap_slots      4
Name: candidate_windows, dtype: int64

names                        [F1_bridge_improvement, F2_roughness_improveme...
compact_names                [F1_bridge_improvement, F3_slope_continuity_im...
epsilon                                                                    0.0
duration_saturation_hours                                                  1.5
robust_bound_scale                                                         3.0
Name: features, dtype: object

## 2. Preflight Validation And Runtime Benchmark

All final-data hashes are checked before an existing partition can be reused.
The benchmark computes real Alpha candidates and projects the clean-run feature
time. It fails before the full scan if the projection exceeds one hour, making
an unexpectedly slow environment visible rather than starting an uncontrolled
job.

In [2]:
hash_audit = validate_input_hashes(PATHS, CONFIG)
display(hash_audit)

alpha_probe = load_dataset("alpha", article_root=ARTICLE_ROOT, config=CONFIG)
alpha_probe = alpha_probe.loc[alpha_probe["substation_id"].eq("alpha_F")]
probe_days = to_day_arrays(alpha_probe)
probe_scale = substation_solar_scale(probe_days.solar[:20], SPEC)

benchmark_started = time.perf_counter()
benchmark_candidates = 0
for day_index in range(20):
    candidate_probe, _ = compute_candidate_features(
        probe_days.net_load[day_index],
        probe_days.solar[day_index],
        substation_solar_scale=probe_scale,
        spec=SPEC,
    )
    benchmark_candidates += len(candidate_probe)
benchmark_seconds = time.perf_counter() - benchmark_started
expected_days = sum(CONFIG["datasets"]["expected_substation_days"].values())
projected_minutes = benchmark_seconds / 20 * expected_days / 60
benchmark = pd.DataFrame(
    [{
        "benchmark_days": 20,
        "mean_candidates_per_day": benchmark_candidates / 20,
        "seconds_per_day": benchmark_seconds / 20,
        "projected_clean_feature_minutes": projected_minutes,
    }]
)
display(benchmark)
assert projected_minutes < 60, (
    f"Projected feature runtime is {projected_minutes:.1f} minutes; optimise or "
    "inspect the environment before launching the full scan."
)
del alpha_probe, probe_days, candidate_probe

,filename,expected_sha256,actual_sha256,matches
0,dataset_alpha.parquet,c2d0982d138a1faebd30df60b7fd861c73007b5709187a...,c2d0982d138a1faebd30df60b7fd861c73007b5709187a...,True
1,dataset_beta.parquet,e1971eff266de145c4aca79ce5272541afee5861d18874...,e1971eff266de145c4aca79ce5272541afee5861d18874...,True
2,dataset_gamma.parquet,7ebeedeab63b1a95059c0f413744230a183bf1d6f6883f...,7ebeedeab63b1a95059c0f413744230a183bf1d6f6883f...,True


,benchmark_days,mean_candidates_per_day,seconds_per_day,projected_clean_feature_minutes
0,20,750.45,0.093006,21.036461


## 3. Cache Design And Leakage Boundary

Each candidate row contains only its substation-day key, window geometry, and
physical features. The candidate file contains no day label, interval label, or
reviewer confidence. A separate day-input cache retains those fields solely for
auditing and later evaluation joins.

F1-F7 are computed directly from observed net load and solar generation. F8 and
F9 use unlabelled within-substation distributions of the daily best
F1+F2+F3 core score. One partition is written per substation, so an interrupted
run can resume without recomputing completed substations. The final consolidated
Parquet file is streamed from those partitions instead of concatenating the
whole candidate universe in memory.

In [3]:
partition_records = []
for dataset in ["alpha", "beta"]:
    dataset_frame = load_dataset(dataset, article_root=ARTICLE_ROOT, config=CONFIG)
    substations = sorted(dataset_frame["substation_id"].unique())
    for substation in substations:
        stem = f"{dataset}_{substation}"
        candidate_path = PARTITION_DIR / f"{stem}_candidates.parquet"
        audit_path = PARTITION_DIR / f"{stem}_day_audit.parquet"
        daily_path = PARTITION_DIR / f"{stem}_daily_best.parquet"
        reusable = (
            CONFIG["execution"]["resume_validated_intermediates"]
            and candidate_path.exists()
            and audit_path.exists()
            and daily_path.exists()
        )

        if not reusable:
            substation_frame = dataset_frame.loc[
                dataset_frame["substation_id"].eq(substation)
            ].copy()
            day_arrays = to_day_arrays(substation_frame, SPEC.slots_per_day)
            candidates, audit, daily_best = build_substation_candidate_features(
                dataset=dataset,
                keys=day_arrays.keys,
                net_load_days=day_arrays.net_load,
                solar_days=day_arrays.solar,
                spec=SPEC,
            )
            write_parquet(candidates, candidate_path)
            write_parquet(audit, audit_path)
            write_parquet(daily_best, daily_path)
            del substation_frame, day_arrays, candidates, audit, daily_best

        candidate_metadata = pq.read_metadata(candidate_path)
        audit_metadata = pq.read_metadata(audit_path)
        partition_records.append(
            {
                "dataset": dataset,
                "substation_id": substation,
                "candidate_path": candidate_path,
                "audit_path": audit_path,
                "daily_path": daily_path,
                "candidate_rows": candidate_metadata.num_rows,
                "substation_days": audit_metadata.num_rows,
                "reused": reusable,
            }
        )
        print(
            f"{dataset} {substation}: {candidate_metadata.num_rows:,} candidates "
            f"across {audit_metadata.num_rows:,} days ({'reused' if reusable else 'built'})",
            flush=True,
        )
    del dataset_frame

partition_index = pd.DataFrame(partition_records)
display(partition_index)

alpha alpha_A: 223,650 candidates across 1,065 days (built)


alpha alpha_B: 815,118 candidates across 1,065 days (built)


alpha alpha_C: 813,790 candidates across 1,065 days (built)


alpha alpha_D: 802,206 candidates across 1,065 days (built)


alpha alpha_E: 817,252 candidates across 1,063 days (built)


...

beta beta_E: 281,072 candidates across 366 days (built)


beta beta_F: 282,066 candidates across 366 days (built)


beta beta_G: 284,740 candidates across 366 days (built)


beta beta_H: 280,095 candidates across 366 days (built)


,dataset,substation_id,candidate_path,audit_path,daily_path,candidate_rows,substation_days,reused
0,alpha,alpha_A,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,223650,1065,False
1,alpha,alpha_B,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,815118,1065,False
2,alpha,alpha_C,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,813790,1065,False
3,alpha,alpha_D,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,802206,1065,False
4,alpha,alpha_E,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,817252,1063,False
5,alpha,alpha_F,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,803046,1063,False
6,alpha,alpha_G,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,807378,1063,False
7,alpha,alpha_H,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,811223,1065,False
8,alpha,alpha_I,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,810697,1064,False
9,alpha,alpha_J,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,C:\Users\z5404477\Documents\PyNRPF\publication...,817572,1065,False


## 4. Consolidate Local Intermediates

The candidate cache is intentionally local-only because it is large but fully
reproducible. Compact tables, notebooks, configuration, figures, and manifests
remain suitable for Git. The day-input cache is also local-only and contains one
row per Alpha or Beta substation-day.

In [4]:
CANDIDATE_CACHE = OUTPUT_DIRS["intermediate"] / "candidate_feature_cache.parquet"
DAY_INPUT_CACHE = OUTPUT_DIRS["intermediate"] / "day_input_cache.parquet"

candidate_parts = partition_index["candidate_path"].tolist()
audit_parts = partition_index["audit_path"].tolist()
consolidate_parquet_files(candidate_parts, CANDIDATE_CACHE)
consolidate_parquet_files(audit_parts, DAY_INPUT_CACHE)

candidate_metadata = pq.read_metadata(CANDIDATE_CACHE)
day_metadata = pq.read_metadata(DAY_INPUT_CACHE)
assert candidate_metadata.num_rows == int(partition_index["candidate_rows"].sum())
assert day_metadata.num_rows == int(partition_index["substation_days"].sum())
assert day_metadata.num_rows == sum(CONFIG["datasets"]["expected_substation_days"].values())

display(
    pd.Series(
        {
            "candidate_rows": candidate_metadata.num_rows,
            "substation_days": day_metadata.num_rows,
            "candidate_cache_GiB": CANDIDATE_CACHE.stat().st_size / 1024**3,
            "day_cache_MiB": DAY_INPUT_CACHE.stat().st_size / 1024**2,
        },
        name="cache",
    )
)

candidate_rows         9.773887e+06
substation_days        1.357100e+04
candidate_cache_GiB    1.747671e-01
day_cache_MiB          1.078825e-01
Name: cache, dtype: float64

## 5. Candidate Counts, Feature Quality, And Label-Join Audit

The first table checks candidate coverage and input gaps by dataset and
substation. The second verifies that every physical feature is finite and
records its observed range. The third proves that each final Alpha/Beta
substation-day has exactly one audit row available for later label joins.

In [5]:
audit_frame = pd.concat(
    [pd.read_parquet(path) for path in partition_index["audit_path"]],
    ignore_index=True,
)
candidate_counts = (
    audit_frame.groupby(["dataset", "substation_id"], as_index=False)
    .agg(
        substation_days=("date", "size"),
        total_candidates=("candidate_count", "sum"),
        minimum_candidates_per_day=("candidate_count", "min"),
        mean_candidates_per_day=("candidate_count", "mean"),
        maximum_candidates_per_day=("candidate_count", "max"),
        input_missing_net_slots=("input_missing_net_slots", "sum"),
        input_missing_solar_slots=("input_missing_solar_slots", "sum"),
        unresolved_net_slots_replaced_with_zero=(
            "unresolved_net_slots_replaced_with_zero", "sum"
        ),
        unresolved_solar_slots_replaced_with_zero=(
            "unresolved_solar_slots_replaced_with_zero", "sum"
        ),
        nonfinite_feature_values_replaced_with_zero=(
            "nonfinite_feature_values_replaced_with_zero", "sum"
        ),
    )
)

feature_quality_rows = []
for record in partition_index.itertuples(index=False):
    features = pd.read_parquet(record.candidate_path, columns=FEATURE_COLUMNS)
    for feature in FEATURE_COLUMNS:
        values = features[feature].to_numpy(dtype=float)
        feature_quality_rows.append(
            {
                "dataset": record.dataset,
                "substation_id": record.substation_id,
                "feature": feature,
                "rows": len(values),
                "finite_rows": int(np.isfinite(values).sum()),
                "minimum": float(np.nanmin(values)),
                "maximum": float(np.nanmax(values)),
                "mean": float(np.nanmean(values)),
            }
        )
feature_quality_detail = pd.DataFrame(feature_quality_rows)
feature_quality = (
    feature_quality_detail.groupby(["dataset", "feature"], as_index=False)
    .agg(
        rows=("rows", "sum"),
        finite_rows=("finite_rows", "sum"),
        minimum=("minimum", "min"),
        maximum=("maximum", "max"),
        mean_of_substation_means=("mean", "mean"),
    )
)
feature_quality["nonfinite_rows"] = feature_quality["rows"] - feature_quality["finite_rows"]
assert feature_quality["nonfinite_rows"].eq(0).all()

expected_days = CONFIG["datasets"]["expected_substation_days"]
join_audit_rows = []
for dataset, expected in expected_days.items():
    subset = audit_frame.loc[audit_frame["dataset"].eq(dataset)]
    duplicate_keys = int(subset.duplicated(["substation_id", "date"]).sum())
    join_audit_rows.append(
        {
            "dataset": dataset,
            "expected_final_substation_days": expected,
            "cached_substation_days": len(subset),
            "duplicate_cache_keys": duplicate_keys,
            "missing_cache_keys": expected - len(subset),
            "positive_day_labels": int(subset["true_day"].sum()),
            "sure_substation_days": int(subset["confidence"].eq("sure").sum()),
            "unsure_substation_days": int(subset["confidence"].eq("unsure").sum()),
        }
    )
join_audit = pd.DataFrame(join_audit_rows)
assert join_audit["missing_cache_keys"].eq(0).all()
assert join_audit["duplicate_cache_keys"].eq(0).all()

COUNTS_PATH = OUTPUT_DIRS["tables"] / "table01_candidate_counts.csv"
QUALITY_PATH = OUTPUT_DIRS["tables"] / "table02_feature_quality_summary.csv"
JOIN_PATH = OUTPUT_DIRS["tables"] / "table03_dataset_and_label_join_audit.csv"
write_csv(candidate_counts, COUNTS_PATH)
write_csv(feature_quality, QUALITY_PATH)
write_csv(join_audit, JOIN_PATH)

display(candidate_counts)
display(feature_quality)
display(join_audit)

,dataset,substation_id,substation_days,total_candidates,minimum_candidates_per_day,mean_candidates_per_day,maximum_candidates_per_day,input_missing_net_slots,input_missing_solar_slots,unresolved_net_slots_replaced_with_zero,unresolved_solar_slots_replaced_with_zero,nonfinite_feature_values_replaced_with_zero
0,alpha,alpha_A,1065,223650,210,210.000000,210,1452,1352,1452,1344,0
1,alpha,alpha_B,1065,815118,210,765.369014,823,780,584,780,576,0
2,alpha,alpha_C,1065,813790,210,764.122066,823,780,584,780,576,0
3,alpha,alpha_D,1065,802206,210,753.245070,823,1740,1544,1740,1536,0
4,alpha,alpha_E,1063,817252,210,768.816557,823,1740,1448,1740,1440,0
5,alpha,alpha_F,1063,803046,210,755.452493,823,1836,1544,1836,1536,0
6,alpha,alpha_G,1063,807378,210,759.527752,823,1644,1352,1644,1344,0
7,alpha,alpha_H,1065,811223,210,761.711737,823,1452,1352,1452,1344,0
8,alpha,alpha_I,1064,810697,210,761.933271,823,1260,1160,1260,1152,0
9,alpha,alpha_J,1065,817572,210,767.673239,823,780,584,780,576,0


,dataset,feature,rows,finite_rows,minimum,maximum,mean_of_substation_means,nonfinite_rows
0,alpha,F1_bridge_improvement,7521932,7521932,-0.999978,0.996494,-0.773456,0
1,alpha,F2_roughness_improvement,7521932,7521932,-0.997257,0.798992,-0.594730,0
2,alpha,F3_slope_continuity_improvement,7521932,7521932,-1.000000,1.000000,-0.290594,0
3,alpha,F4_duration_plausibility,7521932,7521932,0.333333,1.000000,0.934074,0
4,alpha,F5_n_height_ratio,7521932,7521932,0.000000,1.000000,0.033716,0
5,alpha,F6_solar_strength_ratio,7521932,7521932,0.000000,1.000000,0.747802,0
6,alpha,F7_solar_peak_alignment,7521932,7521932,0.000000,1.000000,0.505308,0
7,alpha,F8_substation_centered_core_score,7521932,7521932,-1.000000,1.000000,-0.505633,0
8,alpha,F9_substation_rank_core_score,7521932,7521932,-0.998122,1.000000,0.002871,0
9,beta,F1_bridge_improvement,2251955,2251955,-0.999940,0.982091,-0.686297,0


,dataset,expected_final_substation_days,cached_substation_days,duplicate_cache_keys,missing_cache_keys,positive_day_labels,sure_substation_days,unsure_substation_days
0,alpha,10643,10643,0,0,3423,0,0
1,beta,2928,2928,0,0,630,2310,618


## 6. Findings, Limitations, And Manifest

The cache is complete only if all 13,571 Alpha/Beta substation-days are present,
all F1-F9 values are finite, and no key is duplicated. Physical feature scaling
uses no labels. Reviewer confidence is retained only in the day audit so later
notebooks can restrict training to Beta sure days and report Beta sure versus
Beta all after held-out prediction.

This candidate universe is a development artifact rather than an external test.
Its purpose is to let each later subset or weighted model select its own best
window consistently, resolving the earlier fixed-window cache limitation.

In [6]:
MANIFEST_OUTPUTS = [COUNTS_PATH, QUALITY_PATH, JOIN_PATH]
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[
        PATHS.config,
        PATHS.final_data / "dataset_alpha.parquet",
        PATHS.final_data / "dataset_beta.parquet",
    ],
    outputs=MANIFEST_OUTPUTS,
    row_counts={
        "candidate_rows": candidate_metadata.num_rows,
        "substation_days": day_metadata.num_rows,
        "substation_partitions": len(partition_index),
    },
)
manifest["local_intermediates"] = [
    {
        "path": str(CANDIDATE_CACHE.relative_to(PATHS.article)),
        "bytes": CANDIDATE_CACHE.stat().st_size,
        "git_policy": "local_only_reproducible_cache",
    },
    {
        "path": str(DAY_INPUT_CACHE.relative_to(PATHS.article)),
        "bytes": DAY_INPUT_CACHE.stat().st_size,
        "git_policy": "local_only_reproducible_cache",
    },
]
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)

output_inventory = pd.DataFrame(
    {
        "type": ["candidate cache", "day cache", "table", "table", "table", "manifest"],
        "path": [
            CANDIDATE_CACHE,
            DAY_INPUT_CACHE,
            COUNTS_PATH,
            QUALITY_PATH,
            JOIN_PATH,
            MANIFEST_PATH,
        ],
    }
)
output_inventory["exists"] = output_inventory["path"].map(Path.exists)
output_inventory["bytes"] = output_inventory["path"].map(lambda path: path.stat().st_size)
display(output_inventory)
assert output_inventory["exists"].all()
assert (output_inventory["bytes"] > 0).all()

,type,path,exists,bytes
0,candidate cache,C:\Users\z5404477\Documents\PyNRPF\publication...,True,187654782
1,day cache,C:\Users\z5404477\Documents\PyNRPF\publication...,True,113123
2,table,C:\Users\z5404477\Documents\PyNRPF\publication...,True,1571
3,table,C:\Users\z5404477\Documents\PyNRPF\publication...,True,1732
4,table,C:\Users\z5404477\Documents\PyNRPF\publication...,True,232
5,manifest,C:\Users\z5404477\Documents\PyNRPF\publication...,True,2217
